In [1]:
from collections import Counter
from itertools import combinations
from pathlib import Path
import math

import nltk
import pandas as pd
import regex
import torch
import torch.nn.functional as F
from docx import Document
from nltk.corpus import stopwords

In [2]:
nltk.download("stopwords", quiet=True)

True

In [3]:
def preprocess_text(text, stop_words):
    text = text.lower()
    text = regex.sub(r"[^\p{L}]+", " ", text)
    return [word for word in text.split() if word not in stop_words]

In [4]:
route = Path("docs")
processed_route = Path("docs_processed")
processed_route.mkdir(parents=True, exist_ok=True)

stop_es = set(stopwords.words("spanish"))

for doc in sorted(route.glob("*.docx")):
    paragraphs = [p.text for p in Document(doc).paragraphs if p.text.strip()]
    words = preprocess_text(" ".join(paragraphs), stop_es)

    (processed_route / f"{doc.stem}.txt").write_text(" ".join(words), encoding="utf-8")

print(f"{len(list(processed_route.glob('*.txt')))} documentos preprocesados en '{processed_route}/'")

12 documentos preprocesados en 'docs_processed/'


In [5]:
documents = {}

for doc in sorted(processed_route.glob("*.txt")):
    documents[doc.stem] = doc.read_text(encoding="utf-8").split()

doc_names = list(documents)
vocabulary = sorted({word for words in documents.values() for word in words})

bow_vectors = {}

for doc_name, words in documents.items():
    words_counts = Counter(words)
    bow_vectors[doc_name] = [words_counts[word] for word in vocabulary]

print(f"Documentos: {len(documents)} | Vocabulario: {len(vocabulary)} palabras | Cada documento es un vector de {len(vocabulary)} componentes")

Documentos: 12 | Vocabulario: 411 palabras | Cada documento es un vector de 411 componentes


In [6]:
df = pd.DataFrame(bow_vectors, index=vocabulary)
df.head(10)

,doc01,doc02,doc03,doc04,doc05,doc06,doc07,doc08,doc09,doc10,doc11,doc12
actualizaciones,0,0,0,0,0,0,0,0,0,0,1,0
además,1,0,0,1,0,0,0,0,0,0,1,1
adopción,0,0,0,0,1,0,0,0,1,0,0,0
adquisición,0,0,0,0,1,0,0,0,0,0,0,0
afirman,0,1,0,0,0,0,0,0,0,0,0,0
agigantados,0,0,1,0,0,0,0,0,0,0,0,0
ahora,0,0,0,1,0,0,0,0,0,0,0,0
alta,1,0,0,0,0,0,0,1,0,0,0,0
amantes,1,0,0,0,0,0,0,0,0,0,0,0
ambas,0,0,1,0,0,0,0,0,0,0,0,0


In [7]:
def cosine_pairs(vectors):
    tensors = {name: torch.tensor(vector, dtype=torch.float32) for name, vector in vectors.items()}

    comparaciones = []
    for doc1, doc2 in combinations(tensors, 2):
        similarity = F.cosine_similarity(tensors[doc1], tensors[doc2], dim=0).item()
        comparaciones.append((similarity, doc1, doc2))
    return comparaciones


def similarity_matrix(comparaciones):
    matrix = pd.DataFrame(1.0, index=doc_names, columns=doc_names)
    for similarity, doc1, doc2 in comparaciones:
        matrix.loc[doc1, doc2] = matrix.loc[doc2, doc1] = similarity
    return matrix

In [8]:
comparaciones_bow = cosine_pairs(bow_vectors)

print(f"Pares de documentos comparados: {len(comparaciones_bow)}")
similarity_matrix(comparaciones_bow).round(3)

Pares de documentos comparados: 66


,doc01,doc02,doc03,doc04,doc05,doc06,doc07,doc08,doc09,doc10,doc11,doc12
doc01,1.000,0.044,0.000,0.080,0.099,0.138,0.064,0.015,0.000,0.050,0.146,0.088
doc02,0.044,1.000,0.095,0.000,0.000,0.103,0.124,0.034,0.000,0.111,0.188,0.109
doc03,0.000,0.095,1.000,0.000,0.000,0.033,0.103,0.016,0.000,0.090,0.000,0.053
doc04,0.080,0.000,0.000,1.000,0.100,0.074,0.058,0.129,0.217,0.000,0.136,0.249
doc05,0.099,0.000,0.000,0.100,1.000,0.058,0.000,0.191,0.207,0.042,0.113,0.062
doc06,0.138,0.103,0.033,0.074,0.058,1.000,0.130,0.035,0.017,0.077,0.131,0.103
doc07,0.064,0.124,0.103,0.058,0.000,0.130,1.000,0.018,0.000,0.060,0.190,0.154
doc08,0.015,0.034,0.016,0.129,0.191,0.035,0.018,1.000,0.086,0.019,0.091,0.034
doc09,0.000,0.000,0.000,0.217,0.207,0.017,0.000,0.086,1.000,0.038,0.166,0.323
doc10,0.050,0.111,0.090,0.000,0.042,0.077,0.060,0.019,0.038,1.000,0.071,0.086


In [9]:
a = torch.tensor(bow_vectors["doc01"], dtype=torch.float32)
b = torch.tensor(bow_vectors["doc02"], dtype=torch.float32)

manual = (a @ b) / (a.norm() * b.norm())
pytorch = F.cosine_similarity(a, b, dim=0)

print(f"doc01 vs doc02 -> a mano: {manual.item():.4f} | PyTorch: {pytorch.item():.4f}")

doc01 vs doc02 -> a mano: 0.0441 | PyTorch: 0.0441


In [10]:
comparaciones_bow.sort(key=lambda x: x[0], reverse=True)

for similarity, doc1, doc2 in comparaciones_bow[:10]:
    print(f"Similarity between {doc1} and {doc2}: {similarity:.4f}")

Similarity between doc09 and doc12: 0.3226
Similarity between doc11 and doc12: 0.3016
Similarity between doc04 and doc12: 0.2488
Similarity between doc04 and doc09: 0.2166
Similarity between doc05 and doc09: 0.2067
Similarity between doc05 and doc08: 0.1912
Similarity between doc07 and doc11: 0.1904
Similarity between doc02 and doc11: 0.1883
Similarity between doc09 and doc11: 0.1660
Similarity between doc07 and doc12: 0.1540


In [11]:
sin_palabras_comunes = [(doc1, doc2) for similarity, doc1, doc2 in comparaciones_bow if similarity == 0]

print(f"Pares con similitud 0: {len(sin_palabras_comunes)} de {len(comparaciones_bow)}\n")

for doc1, doc2 in sorted(sin_palabras_comunes):
    print(f"{doc1} y {doc2}")

Pares con similitud 0: 12 de 66

doc01 y doc03
doc01 y doc09
doc02 y doc04
doc02 y doc05
doc02 y doc09
doc03 y doc04
doc03 y doc05
doc03 y doc09
doc03 y doc11
doc04 y doc10
doc05 y doc07
doc07 y doc09


In [12]:
TOPICS = {
    "móvil": ["doc01", "doc02", "doc03", "doc06", "doc07", "doc10"],
    "vehículos eléctricos": ["doc04", "doc05", "doc08", "doc09"],
    "mixto": ["doc11", "doc12"],
}


def mean_similarity(group_a, group_b, comparaciones):
    values = [
        similarity for similarity, doc1, doc2 in comparaciones
        if (doc1 in group_a and doc2 in group_b) or (doc1 in group_b and doc2 in group_a)
    ]
    return sum(values) / len(values)


def topic_table(comparaciones):
    return pd.DataFrame(
        [[mean_similarity(TOPICS[a], TOPICS[b], comparaciones) for b in TOPICS] for a in TOPICS],
        index=TOPICS, columns=TOPICS,
    )

In [13]:
print("Similitud media entre temas (BoW):")
topic_table(comparaciones_bow).round(4)

Similitud media entre temas (BoW):


,móvil,vehículos eléctricos,mixto
móvil,0.0814,0.0252,0.1100
vehículos eléctricos,0.0252,0.1549,0.1466
mixto,0.1100,0.1466,0.3016


In [14]:
def tf_idf(term, document_words, documents):
    if not document_words:
        return 0.0

    tf = document_words.count(term) / len(document_words)

    n_documents_with_term = sum(term in words for words in documents.values())

    idf = math.log(len(documents) / n_documents_with_term)

    return tf * idf

In [15]:
tfidf_vectors = {}

for doc_name, words in documents.items():
    tfidf_vectors[doc_name] = [tf_idf(word, words, documents) for word in vocabulary]

pesos = sorted(zip(vocabulary, tfidf_vectors["doc01"]), key=lambda x: x[1], reverse=True)[:5]

print("Palabras con mayor peso TF-IDF en doc01:")

for palabra, peso in pesos:
    print(f"- {palabra}: {peso:.4f}")

Palabras con mayor peso TF-IDF en doc01:
- tecnophone: 0.1285
- mercado: 0.0857
- xz: 0.0857
- smartphone: 0.0618
- nuevo: 0.0478


In [16]:
comparaciones_tfidf = cosine_pairs(tfidf_vectors)
comparaciones_tfidf.sort(key=lambda x: x[0], reverse=True)

for similarity, doc1, doc2 in comparaciones_tfidf[:10]:
    print(f"Similarity between {doc1} and {doc2}: {similarity:.4f}")

Similarity between doc09 and doc12: 0.1321
Similarity between doc11 and doc12: 0.1171
Similarity between doc04 and doc12: 0.0914
Similarity between doc05 and doc08: 0.0720
Similarity between doc09 and doc11: 0.0715
Similarity between doc05 and doc09: 0.0702
Similarity between doc04 and doc09: 0.0691
Similarity between doc02 and doc11: 0.0670
Similarity between doc02 and doc03: 0.0641
Similarity between doc03 and doc07: 0.0574


In [17]:
print("Similitud media entre temas (TF-IDF):")
topic_table(comparaciones_tfidf).round(4)

Similitud media entre temas (TF-IDF):


,móvil,vehículos eléctricos,mixto
móvil,0.0266,0.0112,0.0279
vehículos eléctricos,0.0112,0.0532,0.0579
mixto,0.0279,0.0579,0.1171


In [18]:
bow_by_pair = {(doc1, doc2): similarity for similarity, doc1, doc2 in comparaciones_bow}
tfidf_by_pair = {(doc1, doc2): similarity for similarity, doc1, doc2 in comparaciones_tfidf}

comparacion_df = pd.DataFrame([
    {
        "documento_1": doc1,
        "documento_2": doc2,
        "similitud_bow": bow_by_pair[(doc1, doc2)],
        "similitud_tfidf": tfidf_by_pair[(doc1, doc2)],
    }
    for doc1, doc2 in bow_by_pair
])

comparacion_df["pos_bow"] = comparacion_df["similitud_bow"].rank(ascending=False, method="min").astype(int)
comparacion_df["pos_tfidf"] = comparacion_df["similitud_tfidf"].rank(ascending=False, method="min").astype(int)

comparacion_df.sort_values("similitud_bow", ascending=False).head(10).round(4).reset_index(drop=True)

,documento_1,documento_2,similitud_bow,similitud_tfidf,pos_bow,pos_tfidf
0,doc09,doc12,0.3226,0.1321,1,1
1,doc11,doc12,0.3016,0.1171,2,2
2,doc04,doc12,0.2488,0.0914,3,3
3,doc04,doc09,0.2166,0.0691,4,7
4,doc05,doc09,0.2067,0.0702,5,6
5,doc05,doc08,0.1912,0.0720,6,4
6,doc07,doc11,0.1904,0.0361,7,20
7,doc02,doc11,0.1883,0.0670,8,8
8,doc09,doc11,0.1660,0.0715,9,5
9,doc07,doc12,0.1540,0.0377,10,19


In [19]:
top_bow = {(doc1, doc2) for _, doc1, doc2 in comparaciones_bow[:10]}
top_tfidf = {(doc1, doc2) for _, doc1, doc2 in comparaciones_tfidf[:10]}

print(f"Pares en común en el top 10: {len(top_bow & top_tfidf)} de 10")
print(f"Solo en el top de BoW:    {sorted(top_bow - top_tfidf)}")
print(f"Solo en el top de TF-IDF: {sorted(top_tfidf - top_bow)}")
print(f"Similitud máxima: BoW = {comparaciones_bow[0][0]:.4f} | TF-IDF = {comparaciones_tfidf[0][0]:.4f}")

Pares en común en el top 10: 8 de 10
Solo en el top de BoW:    [('doc07', 'doc11'), ('doc07', 'doc12')]
Solo en el top de TF-IDF: [('doc02', 'doc03'), ('doc03', 'doc07')]
Similitud máxima: BoW = 0.3226 | TF-IDF = 0.1321
